# PRISM-U Phase 2 — Complete Upload/Resume Colab

This is the **only notebook you need for Phase 2**. It asks for the Phase 2 code ZIP and the required Stage 1 artifacts, verifies the Stage 1 connection, trains/resumes the complete Phase 2 model bank on GPU, freezes the Stage 1 auxiliary query pool, caches black-box probabilities, runs diagnostics, validates completion, saves everything to Google Drive, and exports final result archives.

Before starting: **Runtime → Change runtime type → GPU**. Then run cells from top to bottom. If Colab disconnects later, reopen this notebook, rerun setup/upload cells, and rerun the `RUN / RESUME PHASE 2` cell. Completed work and epoch-resume checkpoints live in Drive.

In [ ]:
# 1) Mount Google Drive and check GPU
from google.colab import drive, files
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
import torch

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/PRISM-U/phase2')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive output:', DRIVE_ROOT)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Use Runtime → Change runtime type → GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))

## 2) Upload the Phase 2 code ZIP
Upload the ZIP downloaded from the GitHub branch **`phase2-upload-package`**. You do not need to rename it. The notebook automatically finds the package root after extraction.

In [ ]:
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one Phase 2 code ZIP in this step.')

UPLOAD_ZIP = Path('/content') / zip_names[0]
UPLOAD_ZIP.write_bytes(uploaded[zip_names[0]])
EXTRACT_ROOT = Path('/content/PRISM-U-Phase2-Code')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(UPLOAD_ZIP, 'r') as zf:
    zf.extractall(EXTRACT_ROOT)

candidates = [p.parent for p in EXTRACT_ROOT.rglob('run_stage2.py') if (p.parent / 'configs/stage2.yaml').is_file()]
if len(candidates) != 1:
    raise RuntimeError(f'Could not uniquely locate Phase 2 package root. Found: {candidates}')
PHASE2_ROOT = candidates[0].resolve()
print('Phase 2 code root:', PHASE2_ROOT)

## 3) Upload the required Stage 1 artifacts
Select these **three files together**: `original_seed42.pt`, `retrain_seed42.pt`, and `cifar10_seed42.json`. You may also include `cifar10_stage1.yaml` so the notebook can cross-check the Stage 1 training configuration. No other Stage 1 outputs are required.

In [ ]:
stage1_upload = files.upload()
required_stage1 = ['original_seed42.pt', 'retrain_seed42.pt', 'cifar10_seed42.json']
missing = [x for x in required_stage1 if x not in stage1_upload]
if missing:
    raise FileNotFoundError('Missing required Stage 1 files: ' + ', '.join(missing))

STAGE1_INPUTS = PHASE2_ROOT / 'stage1_inputs'
STAGE1_INPUTS.mkdir(parents=True, exist_ok=True)
for name, content in stage1_upload.items():
    if name in required_stage1 or name == 'cifar10_stage1.yaml':
        (STAGE1_INPUTS / name).write_bytes(content)
print('Stage 1 inputs copied to:', STAGE1_INPUTS)
print('Files:', sorted(p.name for p in STAGE1_INPUTS.iterdir()))

In [ ]:
# 4) Install dependencies and verify that Phase 2 matches Stage 1
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PHASE2_ROOT / 'requirements.txt')], check=True)

# Optional config cross-check when cifar10_stage1.yaml was uploaded.
if (STAGE1_INPUTS / 'cifar10_stage1.yaml').is_file():
    import yaml
    with open(STAGE1_INPUTS / 'cifar10_stage1.yaml', 'r') as f:
        s1 = yaml.safe_load(f)
    with open(PHASE2_ROOT / 'configs/stage2.yaml', 'r') as f:
        p2 = yaml.safe_load(f)
    fields = ['epochs', 'batch_size', 'learning_rate', 'momentum', 'weight_decay', 'scheduler_milestones', 'scheduler_gamma', 'label_smoothing']
    mismatches = {k: (s1['training'].get(k), p2['training'].get(k)) for k in fields if s1['training'].get(k) != p2['training'].get(k)}
    if s1['model'].get('name') != p2['model'].get('name') or s1['model'].get('num_classes') != p2['model'].get('num_classes') or mismatches:
        raise RuntimeError(f'Phase 2 config does not match uploaded Stage 1 config. Training mismatches: {mismatches}')
    print('Stage 1 YAML cross-check: PASS')

subprocess.run([sys.executable, 'scripts/verify_inputs.py', '--config', 'configs/stage2.yaml'], cwd=PHASE2_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], cwd=PHASE2_ROOT, check=True)
print('\nSTAGE 1 → PHASE 2 CONNECTION: PASS')

# 5) RUN / RESUME PHASE 2
Run this cell. It is intentionally the **single orchestration cell**. It trains all missing surrogate/validation models, creates partial-unlearning checkpoints, builds the final seed-42 family from your Stage 1 models, caches responses, runs diagnostics, and validates completion.

If the session disconnects, rerun cells 1–4 and then rerun this cell. Finished checkpoints are skipped and interrupted training resumes from the epoch state saved in Drive. Per-model black-box responses are also cached in Drive and reused.

In [ ]:
cmd = [sys.executable, 'run_stage2.py', '--config', 'configs/stage2.yaml', '--phase', 'all', '--device', 'cuda']
print('Starting/resuming full PRISM-U Phase 2...')
print('All durable outputs are written continuously to:', DRIVE_ROOT)
subprocess.run(cmd, cwd=PHASE2_ROOT, check=True)
print('\nFull Phase 2 pipeline command finished.')

In [ ]:
# 6) Show the hard completion report
report_path = DRIVE_ROOT / 'validation_report.json'
if not report_path.is_file():
    raise FileNotFoundError('validation_report.json does not exist yet. Re-run the RUN / RESUME cell.')
report = json.loads(report_path.read_text())
print(json.dumps(report, indent=2))
if not report.get('stage2_complete', False):
    raise RuntimeError('Phase 2 is not complete yet. Inspect the failed check above and rerun.')
print('\n' + '=' * 60)
print('PRISM-U PHASE 2 COMPLETE: TRUE')
print('=' * 60)

## 7) Export results
Everything already remains in Drive under `MyDrive/PRISM-U/phase2/`. The cell below also creates:
- a **compact results ZIP** (manifests, query pool, response tensors, diagnostics, reports, logs) and downloads it automatically;
- a **full Phase 2 archive in Drive** including all model checkpoints. The full archive can be several GB, so browser download is optional.

In [ ]:
# Create compact downloadable results (no huge checkpoint bank)
COMPACT_ROOT = Path('/content/PRISM-U-Phase2-Compact-Results')
if COMPACT_ROOT.exists():
    shutil.rmtree(COMPACT_ROOT)
COMPACT_ROOT.mkdir(parents=True)
for name in ['manifests', 'query_pool', 'responses', 'diagnostics', 'reports', 'logs']:
    src = DRIVE_ROOT / name
    if src.exists():
        shutil.copytree(src, COMPACT_ROOT / name)
if (DRIVE_ROOT / 'validation_report.json').exists():
    shutil.copy2(DRIVE_ROOT / 'validation_report.json', COMPACT_ROOT / 'validation_report.json')
compact_zip = shutil.make_archive('/content/PRISM-U-Phase2-Results', 'zip', root_dir=COMPACT_ROOT)
drive_compact = DRIVE_ROOT / 'PRISM-U-Phase2-Results.zip'
shutil.copy2(compact_zip, drive_compact)
print('Compact results saved to Drive:', drive_compact)
files.download(compact_zip)

# Full archive including ~100 checkpoint files. Saved to Drive; downloading it is optional because it may be several GB.
FULL_ARCHIVE_BASE = Path('/content/drive/MyDrive/PRISM-U/PRISM-U-Phase2-FULL')
full_zip = shutil.make_archive(str(FULL_ARCHIVE_BASE), 'zip', root_dir=DRIVE_ROOT)
print('Full Phase 2 archive saved to Drive:', full_zip)
print('All individual checkpoints also remain uncompressed at:', DRIVE_ROOT / 'checkpoints')

DOWNLOAD_FULL_ARCHIVE = False  # set True only if you really want a multi-GB browser download
if DOWNLOAD_FULL_ARCHIVE:
    files.download(full_zip)